In [45]:
import pandas as pd
import altair as alt
import ipywidgets as widgets
from IPython.display import display, clear_output
from ta.momentum import StochasticOscillator

In [46]:
df = pd.read_csv("5companies_yahoofinance.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['ticker', 'Date'])

def compute_kd_index(group):
    stoch = StochasticOscillator(
        high=group['High'],
        low=group['Low'],
        close=group['Close']
    )
    group['K_percent'] = stoch.stoch()
    group['D_percent'] = stoch.stoch_signal()
    return group

df = df.groupby('ticker').apply(compute_kd_index).reset_index(drop=True)

companies_dropdown = widgets.Dropdown(
    options=['AAPL', 'NVDA', 'AMD', 'QCOM', 'TSM'],
    value='AAPL',
    description='Company: '
)

In [47]:
date_slider = widgets.SelectionRangeSlider(
    options=sorted(df['Date'].dt.strftime('%Y-%m-%d').unique().tolist()),
    index=(0, len(df['Date'].unique()) - 1),
    description='Date Range:',
    layout={'width': '800px'}
)

k_toggle = widgets.Checkbox(value=True, description='Show %K (green)')
d_toggle = widgets.Checkbox(value=True, description='Show %D (red)')

In [48]:
# Plot function
def plot_company_stockprice(ticker, date_range, show_k, show_d):
    df_filtered = df[
        (df['ticker'] == ticker) &
        (df['Date'] >= pd.to_datetime(date_range[0])) &
        (df['Date'] <= pd.to_datetime(date_range[1]))
    ]
    
    stock_price_chart = alt.Chart(df_filtered).mark_line().encode(
        x=alt.X('Date:T', title='Date'),
        y=alt.Y('Close:Q', title='Close Price'),
        tooltip=['Date:T', 'Close:Q', 'ticker:N'],
    ).properties(
        width=600,
        height=400
    )
    
    kd_lines = []
    if show_k:
        k_line = alt.Chart(df_filtered).mark_line(color='green').encode(
            x='Date:T',
            y=alt.Y('K_percent:Q', title='K / D (%)'),
            tooltip=['Date:T', 'K_percent:Q']
        )
        kd_lines.append(k_line)
    
    if show_d:
        d_line = alt.Chart(df_filtered).mark_line(color='red').encode(
            x='Date:T',
            y='D_percent:Q',
            tooltip=['Date:T', 'D_percent:Q']
        )
        kd_lines.append(d_line)
    
    kd_chart = alt.layer(*kd_lines).properties(height=150)

    return alt.vconcat(stock_price_chart, kd_chart).resolve_scale(x='shared')


In [49]:
output = widgets.Output()

def update_company_plot(change=None):
    with output:
        clear_output()
        display(plot_company_stockprice(
            companies_dropdown.value,
            date_slider.value,
            k_toggle.value,
            d_toggle.value
        ))

companies_dropdown.observe(update_company_plot, names='value')
date_slider.observe(update_company_plot, names='value')
k_toggle.observe(update_company_plot, names='value')
d_toggle.observe(update_company_plot, names='value')

display(widgets.VBox([
    companies_dropdown,
    date_slider,
    widgets.HBox([k_toggle, d_toggle]),
    output
]))

update_company_plot()

Date range slider & Checkbox toggles for %K and %D lines